In [1]:
from battery_sizing_cfa.cost_functions.lcoe import lcoe_from_results
from battery_sizing_cfa.optimizers.deterministic_sizing import optimize_lcoe_prescient
from battery_sizing_cfa.optimizers.parametric_rbc import rbc
import numpy as np
import plotly.graph_objects as go
from pyomo.contrib.appsi.solvers.highs import Highs as AppsiHighs


# Define synthetic profiles
N = 24*14
Delta_t = 1.0
L = np.array([40 + 10*np.sin(2*np.pi*(t-7)/24) + 50*(np.sin(2*np.pi*(t-7)/(24*7))/((t/10)%(24*7/10)+3)) for t in range(N)])          # MW
PV_base = np.maximum(0, 60*np.sin(np.pi*(np.arange(N)/12 - 0.5)))           # MW
price = np.array([40 if (t%24) in range(18,24) else 20 for t in range(N)])  # $/MWh
export_price = np.ones(N)*0.1



# Plot synthetic profiles
fig = go.Figure()
time_steps = np.arange(N)
fig.add_trace(go.Scatter(x=time_steps, y=L, mode='lines', name='Load (MW)'))
fig.add_trace(go.Scatter(x=time_steps, y=PV_base, mode='lines', name='Base PV (MW)'))
fig.add_trace(go.Scatter(x=time_steps, y=price, mode='lines', name='Price ($/MWh)'))

KeyboardInterrupt: 

In [ ]:
specs = {'c_PV_kw': 200.0,
         'c_bat_E_kWh': 120.0,
         'c_bat_P_kw': 50.0,
         'eta_ch': 0.95,
         'eta_dis': 0.95,
         'cyclic_SOC': True,
         'peak_tariff_per_MW_period': 100.0,
         'peak_period_steps': 24,
         'discount_rate': 0.06,
         'lifetime_years': 15,
         'replicate_periods': 26,
         'Delta_t':Delta_t,
         'soc_min':0,
         'soc_max':1,
         'soc_start':0.5,
         'energy_ratio':0.5
         }

res = optimize_lcoe_prescient(
    L=L, PV_base=PV_base, price=price, export_price=export_price,
    solver_options={"time_limit": 120, "threads": 2}, **specs
)

print("Solver:", res["solver"], "| Termination:", res["termination"])
print("LCOE [$ / MWh]:", round(res["LCOE_$/MWh"], 2)) # Keep LCOE in $/MWh as it is a standard metric
print("x_pv:", round(res["x_pv"],3), "E_bat [kWh]:", round(res["E_bat_kWh"],2), "P_bat_max [kW]:", round(res["P_bat_max_kW"],2))
if "period_peaks_MW" in res: # Keep period_peaks_MW as it is used for the tariff
    print("Daily peak import [MW]:", np.round(res["period_peaks_MW"], 2))

In [ ]:
from utils.plot_utils import plot_results

plot_results(res['x_pv'], res["E_bat_kWh"], res["series"]['P_ch_kW']-res["series"]['P_dis_kW'], res["series"]["SOC_kWh"], PV_base, L, price)

In [ ]:
calculated_lcoe = lcoe_from_results(
    L=L,
    PV_base=PV_base,
    price=price,
    export_price=export_price,
    x_pv=res["x_pv"],
    E_bat_kWh=res["E_bat_kWh"],
    P_bat_max_kW=res["P_bat_max_kW"],
    P_net_kW=res["series"]["P_net_kW"],
    period_peaks_MW=res.get("period_peaks_MW", None),
    **specs
)
print("LCOE from optimization (from optimizer) [$ / MWh]:", round(res["LCOE_$/MWh"], 2))
print("LCOE from optimization (ex-post from results) [$ / MWh]:", round(calculated_lcoe, 2))

In [ ]:
price_high = price == max(price)

soc_sim, p_batt_sim, p_grid_sim = rbc(
    p_tot=L - res["x_pv"] * PV_base,  # net load is total consumption
    price_high=price_high,
    capacity_kwh=float(res["E_bat_kWh"]), # Use kWh directly
    soc_min_price_fun=np.array([0.50]),
    p_threshold_fun=np.array([30.0]),
    p_charge_max=float(res["P_bat_max_kW"]), # Use kW directly
    p_discharge_max=float(res["P_bat_max_kW"]), # Use kW directly
    eta_ch=specs['eta_ch'], eta_dis=specs['eta_dis'],
    dt_hours=specs['Delta_t'],
    soc_start=specs['soc_start'],
    soc_min=specs['soc_min'],
    soc_max=specs['soc_max'])


lcoe_simulation = lcoe_from_results(
    L=L,
    PV_base=PV_base,
    price=price,
    export_price=export_price,
    x_pv=res["x_pv"],  # Use the PV size from optimization
    E_bat_kWh=res["E_bat_kWh"], # Use battery energy capacity from optimization
    P_bat_max_kW=res["P_bat_max_kW"], # Use battery power capacity from optimization
    P_net_kW=p_grid_sim, # Use the net load from the simulation
    period_peaks_MW=[np.max(p_grid_sim[i*24:(i+1)*24])/1000.0 for i in range(N//24) if len(p_grid_sim[i*24:(i+1)*24]) > 0],
    **specs
)
print("LCOE from RBC [$ / MWh]:", round(lcoe_simulation, 2))
plot_results(res['x_pv'], res["E_bat_kWh"], p_batt_sim, soc_sim*res["E_bat_kWh"], PV_base, L, price)


In [ ]:
from battery_sizing_cfa.optimizers.rbc_sizing import  rbc_opt_fun

# Define the range for p_threshold and soc_min_price
p_threshold_range = np.linspace(0, 60, 50) # Adjust the range as needed
soc_min_price_range = np.linspace(0, 1, 50)   # Adjust the range as needed

# Create a grid of parameter values
P_THRESHOLD, SOC_MIN_PRICE = np.meshgrid(p_threshold_range, soc_min_price_range)

# Initialize a list to store LCOE values
lcoe_values = []

# Compute LCOE for each combination of parameters
for i in range(P_THRESHOLD.shape[0]):
    row_lcoe = []
    for j in range(P_THRESHOLD.shape[1]):
        params = [res['x_pv'], res["E_bat_kWh"], P_THRESHOLD[i, j], SOC_MIN_PRICE[i, j]]
        lcoe = rbc_opt_fun(
            params,
            L=L,
            PV_base=PV_base,
            price=price,
            export_price=export_price,
            specs=specs, noise_level=0
        )
        row_lcoe.append(lcoe)
    lcoe_values.append(row_lcoe)

LCOE_SURFACE = np.array(lcoe_values)

# Create the surface plot
fig = go.Figure(data=[go.Surface(z=LCOE_SURFACE, x=P_THRESHOLD, y=SOC_MIN_PRICE)])

fig.update_layout(
    title='LCOE Landscape',
    scene = dict(
        xaxis_title='P_threshold (kW)',
        yaxis_title='SOC_min_price',
        zaxis_title='LCOE ($/MWh)'),
    autosize=False,
    width=800,
    height=700,
    margin=dict(l=65, r=50, b=65, t=90)
)

fig.show()

In [ ]:
# Define the range for p_treshold and soc_min_price
battery_size = np.linspace(0, 100, 50) # Adjust the range as needed
pv_size = np.linspace(0, 2, 50)   # Adjust the range as needed



# Create a grid of parameter values
B_size, PV_size = np.meshgrid(battery_size, pv_size)

# Initialize a list to store LCOE values
lcoe_values = []

# Compute LCOE for each combination of parameters
for i in range(B_size.shape[0]):
    row_lcoe = []
    for j in range(B_size.shape[1]):
        params = [PV_size[i, j], B_size[i, j], 30., 1.]
        lcoe = rbc_opt_fun(params,
            L=L,
            PV_base=PV_base,
            price=price,
            export_price=export_price,
            specs=specs
        )
        row_lcoe.append(lcoe)
    lcoe_values.append(row_lcoe)

LCOE_SURFACE = np.array(lcoe_values)

# Create the surface plot
fig = go.Figure(data=[go.Surface(z=LCOE_SURFACE, x=B_size, y=PV_size)])

fig.update_layout(
    title='LCOE Landscape',
    scene = dict(
        xaxis_title='Battery size (kWh)',
        yaxis_title='PV size (kW)',
        zaxis_title='LCOE ($/MWh)'),
    autosize=False,
    width=800,
    height=700,
    margin=dict(l=65, r=50, b=65, t=90)
)

fig.show()

In [ ]:
from scipy.optimize import differential_evolution

bounds = [
    (0, 3*res['x_pv']),         # x_pv
    (0, 3*res["E_bat_kWh"]),    # E_bat_kWh
    (0, 100),                   # p_threshold
    (0, 1)                      # soc_min_price
]

result = differential_evolution(
    rbc_opt_fun,
    bounds,
    args=(L, PV_base, price, export_price, specs),
    init='random',
    strategy='best1bin',
    maxiter=100,
    popsize=15,
    tol=0.01,
    polish=False
)

print("Optimized parameters (x_pv, E_bat_kWh, p_threshold, soc_min_price):", result.x)
print("Minimum LCOE from RBC optimization [$ / MWh]:", result.fun)

# Learn conditional parameters
We can make p_threshold_range and soc_min_price_range functions of the state of charge or time of day to learn more complex policies.

In [ ]:
from battery_sizing_cfa.optimizers.rbc_sizing import  rbc_opt_fun

# Define the range for p_threshold and soc_min_price
p_threshold_range = np.linspace(0, 60, 50) # Adjust the range as needed
soc_min_price_range = np.linspace(0, 1, 50)   # Adjust the range as needed

# Create a grid of parameter values
P_THRESHOLD, SOC_MIN_PRICE = np.meshgrid(p_threshold_range, soc_min_price_range)

# Initialize a list to store LCOE values
lcoe_values = []

# Compute LCOE for each combination of parameters
for i in range(P_THRESHOLD.shape[0]):
    row_lcoe = []
    for j in range(P_THRESHOLD.shape[1]):
        params = [res['x_pv'], res["E_bat_kWh"], P_THRESHOLD[i, j], SOC_MIN_PRICE[i, j]]
        lcoe = rbc_opt_fun(
            params,
            L=L,
            PV_base=PV_base,
            price=price,
            export_price=export_price,
            specs=specs, noise_level=0
        )
        row_lcoe.append(lcoe)
    lcoe_values.append(row_lcoe)

LCOE_SURFACE = np.array(lcoe_values)

# Create the surface plot
fig = go.Figure(data=[go.Surface(z=LCOE_SURFACE, x=P_THRESHOLD, y=SOC_MIN_PRICE)])

fig.update_layout(
    title='LCOE Landscape',
    scene = dict(
        xaxis_title='P_threshold (kW)',
        yaxis_title='SOC_min_price',
        zaxis_title='LCOE ($/MWh)'),
    autosize=False,
    width=800,
    height=700,
    margin=dict(l=65, r=50, b=65, t=90)
)

fig.show()